## 1. A simple self-attention mechanism without trainable weights

1. Calculate the *attention scores* by taking the dot product of the input vectors with a *query* vector or with each other.
2. Apply the softmax function to the attention scores to get the *attention weights*.
3. Compute the *context vector* as a weighted sum of the input vectors, using the attention weights as coefficients.

In [2]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [ ]:
query = inputs[1]
attention_score = torch.matmul(inputs, query)
print(attention_score)

x_size = inputs.shape[0]
attention_score_2 = torch.empty(x_size)
for i in range(x_size):
  attention_score_2[i] = torch.dot(inputs[i], query)
print(attention_score_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [ ]:
# normalize attention scores to get attention weights

attention_weights = torch.softmax(attention_score, dim=0)
print(attention_weights)

attention_weights_2 = attention_score_2 / attention_score_2.sum()
print(attention_weights_2)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])


In [ ]:
context_vector = torch.matmul(attention_weights, inputs)
print(context_vector)

context_vector_2 = torch.zeros(query.shape)
for i in range(x_size):
  context_vector_2 += attention_weights[i] * inputs[i]
print(context_vector_2)


tensor([0.4419, 0.6515, 0.5683])
tensor([0.4419, 0.6515, 0.5683])


In [ ]:
# apply to all input vectors

attention_scores = torch.matmul(inputs, inputs.T)
# `dim=-1` apply softmax to last dimension. e.g. for 2D tensor, it will normalize columns so that each row sums to 1
attention_weights = torch.softmax(attention_scores, dim=-1)
print(attention_weights)
print(attention_weights.sum(dim=-1))

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [19]:
context_vectors = torch.matmul(attention_weights, inputs)
print(context_vectors)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


## 2. Implement self-attention with trainable weights

This self-attention mechanism is also called "scaled dot-product attention", which is the core of the Transformer architecture.

Each input vector is transformed into a query, key, and value vector using trainable weight matrices $W_q$, $W_k$, and $W_v$.

1. Compute the query, key, and value vectors for each input vector.
2. Calculate the attention scores by taking the dot product of the query vector with the key vectors.
3. Scale the attention scores by dividing by the square root of the dimension of the key vectors. Then apply the softmax function to get the attention weights.
    - large inputs can make the softmax function produce very small gradients, which can lead to vanishing gradients during training. Scaling the attention scores helps produce more stable gradients.
4. Compute the context vector as a weighted sum of the value vectors, using the attention weights as coefficients.

In [7]:
# first, computing only one context vector
x = inputs[1]
d_in = inputs.shape[1]
d_out = 2

torch.manual_seed(42)
W_q = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad=False)
W_k = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad=False)
W_v = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad=False)

In [ ]:
query = x @ W_q
key = x @ W_k
value = x @ W_v

# compute keys and values for all input vectors
keys = inputs @ W_k
values = inputs @ W_v

In [31]:
attention_scores_2 = query @ keys.T
print(attention_scores_2)

tensor([-0.4540, -0.6313, -0.6450, -0.2855, -0.7087, -0.1794])


In [ ]:
# scale attention scores
attention_weights_2 = torch.softmax(attention_scores_2 / keys.shape[-1] ** 0.5, dim=-1)
attention_weights_2

tensor([0.1686, 0.1487, 0.1473, 0.1899, 0.1408, 0.2047])

In [43]:
context_vector_2 = attention_weights_2 @ values
print(context_vector_2)

tensor([0.5633, 0.3251])


In [ ]:
import torch.nn as nn

class SelfAttentionV1(nn.Module):
  def __init__(self, d_in, d_out):
    super().__init__()
    self.q = nn.Parameter(torch.randn(d_in, d_out))
    self.k = nn.Parameter(torch.randn(d_in, d_out))
    self.v = nn.Parameter(torch.randn(d_in, d_out))

  def forward(self, x):
    query = x @ self.q
    keys = x @ self.k
    values = x @ self.v

    attention_scores = query @ keys.T
    attention_weights = torch.softmax(attention_scores / keys.shape[-1] ** 0.5, dim=-1)

    context_vectors = attention_weights @ values
    return context_vectors

In [53]:
torch.manual_seed(42)
sa_v1 = SelfAttentionV1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.5141, 0.3639],
        [0.5633, 0.3251],
        [0.5659, 0.3221],
        [0.5839, 0.2941],
        [0.6180, 0.2539],
        [0.5575, 0.3262]], grad_fn=<MmBackward0>)


In [ ]:
import torch.nn as nn

class SelfAttentionV2(nn.Module):
  """A simple implementation of self-attention mechanism."""
  def __init__(self, d_in: int, d_out: int, bias: bool = False):
    super().__init__()
    self.d_out = d_out
    # learnable linear transformations for query, key, and value
    self.q = nn.Linear(d_in, d_out, bias=bias)
    self.k = nn.Linear(d_in, d_out, bias=bias)
    self.v = nn.Linear(d_in, d_out, bias=bias)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    query = self.q(x)
    keys = self.k(x)
    values = self.v(x)

    attention_scores = query @ keys.T
    attention_weights = torch.softmax(attention_scores / self.d_out ** 0.5, dim=-1)

    context_vectors = attention_weights @ values
    return context_vectors

In [8]:
torch.manual_seed(42)
sa_v2 = SelfAttentionV2(d_in, d_out)
print(sa_v2(inputs))

tensor([[0.3755, 0.2777],
        [0.3761, 0.2831],
        [0.3761, 0.2833],
        [0.3768, 0.2763],
        [0.3754, 0.2836],
        [0.3772, 0.2746]], grad_fn=<MmBackward0>)


In [ ]:
sa_v1.q = nn.Parameter(sa_v2.q.weight.T)
sa_v1.k = nn.Parameter(sa_v2.k.weight.T)
sa_v1.v = nn.Parameter(sa_v2.v.weight.T)
sa_v1(inputs)

tensor([[0.3755, 0.2777],
        [0.3761, 0.2831],
        [0.3761, 0.2833],
        [0.3768, 0.2763],
        [0.3754, 0.2836],
        [0.3772, 0.2746]], grad_fn=<MmBackward0>)

## 3. Implementing causal attention

Causal attention is a variant of self-attention that prevents the model from attending to future tokens in the sequence.

In [ ]:
# calculate masked attention weights
queries = sa_v2.q(inputs)
keys = sa_v2.k(inputs)
attention_scores = queries @ keys.T
attention_weights = torch.softmax(attention_scores / keys.shape[-1] ** 0.5, dim=-1)

# create a mask where upper triangular part (excluding the diagonal) is filled with 0
seq_len = inputs.shape[0]
mask = torch.tril(torch.ones(seq_len, seq_len))

# apply the mask to attention weights
masked_weights = attention_weights * mask
row_sums = masked_weights.sum(dim=-1, keepdim=True)
masked_weights_norm = masked_weights / row_sums
print(masked_weights_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4775, 0.5225, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3146, 0.3450, 0.3405, 0.0000, 0.0000, 0.0000],
        [0.2459, 0.2555, 0.2538, 0.2448, 0.0000, 0.0000],
        [0.1969, 0.2193, 0.2165, 0.2053, 0.1619, 0.0000],
        [0.1682, 0.1715, 0.1707, 0.1648, 0.1511, 0.1738]],
       grad_fn=<DivBackward0>)


In [ ]:
# more efficient way to apply the mask directly to attention scores before softmax

# create a mask where upper triangular part (excluding the diagonal) is filled with 1
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
# replace `1` in mask with `-inf` so that their softmax will be 0
masked_scores = attention_scores.masked_fill(mask.bool(), -torch.inf)
print(masked_scores)
masked_attention_weights = torch.softmax(masked_scores / keys.shape[-1] ** 0.5, dim=-1)
print(masked_attention_weights)

tensor([[ 0.0508,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.2157,  0.3428,    -inf,    -inf,    -inf,    -inf],
        [ 0.2163,  0.3467,  0.3282,    -inf,    -inf,    -inf],
        [ 0.1257,  0.1799,  0.1707,  0.1191,    -inf,    -inf],
        [ 0.1667,  0.3193,  0.3012,  0.2258, -0.1098,    -inf],
        [ 0.1269,  0.1548,  0.1475,  0.0978, -0.0247,  0.1731]],
       grad_fn=<MaskedFillBackward0>)
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4775, 0.5225, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3146, 0.3450, 0.3405, 0.0000, 0.0000, 0.0000],
        [0.2459, 0.2555, 0.2538, 0.2448, 0.0000, 0.0000],
        [0.1969, 0.2193, 0.2165, 0.2053, 0.1619, 0.0000],
        [0.1682, 0.1715, 0.1707, 0.1648, 0.1511, 0.1738]],
       grad_fn=<SoftmaxBackward0>)


In [ ]:
# dropout will randomly set some of the elements to zero， it is a technique to prevent overfitting
# elements that are not set to zero will be scaled up by `1/(1-p)` to keep the expected sum of the outputs.
torch.manual_seed(42)
dropout = nn.Dropout(p=0.5)
sample = torch.ones((4, 4))
print(dropout(sample))

tensor([[2., 2., 2., 2.],
        [0., 2., 0., 0.],
        [2., 2., 2., 2.],
        [0., 0., 2., 0.]])


In [39]:
dropout(masked_attention_weights)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.9551, 1.0449, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6292, 0.6899, 0.6809, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.4895, 0.0000, 0.0000],
        [0.0000, 0.4387, 0.4331, 0.4106, 0.0000, 0.0000],
        [0.3364, 0.3431, 0.0000, 0.0000, 0.0000, 0.3476]],
       grad_fn=<MulBackward0>)

In [ ]:
class CasualSelfAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout=0.0, bias=False):
        super().__init__()
        self.d_out = d_out
        self.q = nn.Linear(d_in, d_out, bias=bias)
        self.k = nn.Linear(d_in, d_out, bias=bias)
        self.v = nn.Linear(d_in, d_out, bias=bias)
        self.dropout = nn.Dropout(p=dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        batch_size, seq_len, d_in = x.shape
        query = self.q(x)
        keys = self.k(x)
        values = self.v(x)

        attention_scores: torch.Tensor = query @ keys.transpose(-2, -1)
        # trim the mask to the current sequence length and apply it to attention scores
        attention_scores.masked_fill_(
            self.mask.bool()[:seq_len, :seq_len], -torch.inf
        )

        attention_weights = torch.softmax(attention_scores / self.d_out ** 0.5, dim=-1)
        attention_weights = self.dropout(attention_weights)

        context_vectors = attention_weights @ values
        return context_vectors

In [44]:
batch = torch.stack([inputs, inputs])  # shape: (batch_size, seq_len, d_in)
print(batch.shape)

torch.Size([2, 6, 3])


In [61]:
torch.manual_seed(42)
csa = CasualSelfAttention(d_in, d_out, context_length=batch.shape[1])
output = csa(batch)
print(output.shape)
print(output)

torch.Size([2, 6, 2])
tensor([[[0.4429, 0.1077],
         [0.4656, 0.2597],
         [0.4732, 0.3030],
         [0.4135, 0.2921],
         [0.4078, 0.2567],
         [0.3772, 0.2746]],

        [[0.4429, 0.1077],
         [0.4656, 0.2597],
         [0.4732, 0.3030],
         [0.4135, 0.2921],
         [0.4078, 0.2567],
         [0.3772, 0.2746]]], grad_fn=<UnsafeViewBackward0>)


## 4. Multi-head attention

Multi-head attention allows the model to attend to different parts of the input sequence simultaneously by using multiple sets of query, key, and value weight matrices. Each set is called a "head". The outputs of all heads are concatenated and projected back to the original dimension.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_length, dropout=0.0, bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.q = nn.Linear(d_in, d_out, bias=bias)
        self.k = nn.Linear(d_in, d_out, bias=bias)
        self.v = nn.Linear(d_in, d_out, bias=bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(p=dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        batch_size, seq_len, d_in = x.shape
        query = self.q(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        keys = self.k(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        values = self.v(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        # reshape to (batch_size, num_heads, seq_len, head_dim)
        query = query.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        attention_scores = query @ keys.transpose(-2, -1) / (self.head_dim**0.5)
        attention_scores.masked_fill_(self.mask.bool()[:seq_len, :seq_len], -torch.inf)

        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        context_vectors = attention_weights @ values
        # reshape back to (batch_size, seq_len, d_out)
        context_vectors = context_vectors.transpose(1, 2).contiguous()
        context_vectors = context_vectors.view(batch_size, seq_len, self.d_out)

        output = self.out_proj(context_vectors)
        return output

In [65]:
torch.manual_seed(42)

mha = MultiHeadAttention(d_in, d_out, num_heads=2, context_length=batch.shape[1])
output = mha(batch)
print(output.shape)
print(output)

torch.Size([2, 6, 2])
tensor([[[-0.4140,  0.3158],
         [-0.3942,  0.2961],
         [-0.3888,  0.2900],
         [-0.3714,  0.3100],
         [-0.3764,  0.3150],
         [-0.3632,  0.3238]],

        [[-0.4140,  0.3158],
         [-0.3942,  0.2961],
         [-0.3888,  0.2900],
         [-0.3714,  0.3100],
         [-0.3764,  0.3150],
         [-0.3632,  0.3238]]], grad_fn=<ViewBackward0>)
